# Evolent Health

In [27]:
# libaries
import pandas as pd

# import data
df = pd.read_csv(
    'data/BeerDataScienceProject.tar.bz2',
    compression='bz2')
df.head()

,beer_ABV,beer_beerId,beer_brewerId,beer_name,beer_style,review_appearance,review_palette,review_overall,review_taste,review_profileName,review_aroma,review_text,review_time
0,5.0,47986,10325,Sausa Weizen,Hefeweizen,2.5,2.0,1.5,1.5,stcules,1.5,A lot of foam. But a lot. In the smell some ba...,1234817823
1,6.2,48213,10325,Red Moon,English Strong Ale,3.0,2.5,3.0,3.0,stcules,3.0,"Dark red color, light beige foam, average. In ...",1235915097
2,6.5,48215,10325,Black Horse Black Beer,Foreign / Export Stout,3.0,2.5,3.0,3.0,stcules,3.0,"Almost totally black. Beige foam, quite compac...",1235916604
3,5.0,47969,10325,Sausa Pils,German Pilsener,3.5,3.0,3.0,2.5,stcules,3.0,"Golden yellow color. White, compact foam, quit...",1234725145
4,7.7,64883,1075,Cauldron DIPA,American Double / Imperial IPA,4.0,4.5,4.0,4.0,johnmichaelsen,4.5,"According to the website, the style for the Ca...",1293735206


In [28]:
# clean review_time
df['review_time'] = pd.to_datetime(df['review_time'], unit='s')
df.head()

,beer_ABV,beer_beerId,beer_brewerId,beer_name,beer_style,review_appearance,review_palette,review_overall,review_taste,review_profileName,review_aroma,review_text,review_time
0,5.0,47986,10325,Sausa Weizen,Hefeweizen,2.5,2.0,1.5,1.5,stcules,1.5,A lot of foam. But a lot. In the smell some ba...,2009-02-16 20:57:03
1,6.2,48213,10325,Red Moon,English Strong Ale,3.0,2.5,3.0,3.0,stcules,3.0,"Dark red color, light beige foam, average. In ...",2009-03-01 13:44:57
2,6.5,48215,10325,Black Horse Black Beer,Foreign / Export Stout,3.0,2.5,3.0,3.0,stcules,3.0,"Almost totally black. Beige foam, quite compac...",2009-03-01 14:10:04
3,5.0,47969,10325,Sausa Pils,German Pilsener,3.5,3.0,3.0,2.5,stcules,3.0,"Golden yellow color. White, compact foam, quit...",2009-02-15 19:12:25
4,7.7,64883,1075,Cauldron DIPA,American Double / Imperial IPA,4.0,4.5,4.0,4.0,johnmichaelsen,4.5,"According to the website, the style for the Ca...",2010-12-30 18:53:26


# Question 1
**Rank the top 3 breweries which produce the strongest beers.**

In [29]:
# copy df
q1 = df.copy()
# group by brewer, calculate mean ABV
q1 = q1.groupby('beer_brewerId')['beer_ABV'].mean().reset_index()
# rename column
q1.rename(columns={'beer_ABV':'mean_ABV'}, inplace=True)
# get 3 largest
q1.nlargest(columns='mean_ABV', n=3)

,beer_brewerId,mean_ABV
784,6513,19.228824
175,736,13.750000
1644,24215,12.466667


# Question 2
**Which year did beers enjoy the highest ratings?**

In [30]:
# copy df
q2 = df.copy()
# create year column
q2['year'] = q2['review_time'].dt.year
q2 = q2.groupby('year')['review_overall'].agg(['count', 'mean']).round(3).rename(columns={'count':'n reviews', 'mean':'mean_overall'}).sort_values('mean_overall', ascending=False).reset_index()
q2

,year,n reviews,mean_overall
0,2000,33,4.182
1,1999,25,4.000
2,2001,602,3.928
3,1998,23,3.891
4,2010,93810,3.866
5,2009,83578,3.864
6,2008,69080,3.834
7,2005,29433,3.832
8,2012,3180,3.830
9,2011,110836,3.828


In [31]:
# Filter for years where n reviews is more 10,000
q2.loc[
    (q2['n reviews'] > 10_000)
].head(1)

,year,n reviews,mean_overall
4,2010,93810,3.866


# Question 3
**Based on the users' ratings, which factors are important among taste, aroma, appearance, and palette?**

In [32]:
# copy df
q3 = df.copy()

# initialize random forest
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor()

# features
X = q3[['review_taste', 'review_aroma', 'review_appearance', 'review_palette']]
# target
y = q3['review_overall']
# fit model
model.fit(X,y)
# feature and their importances
model_coef_df = pd.DataFrame({
    'feature':model.feature_names_in_,
    'importances':model.feature_importances_
    })
model_coef_df.sort_values('importances', ascending=False)

,feature,importances
1,review_aroma,0.926431
0,review_taste,0.055504
2,review_appearance,0.009616
3,review_palette,0.008449


In [33]:
model_coef_df.nlargest(columns='importances', n=1)

,feature,importances
1,review_aroma,0.926431


# Question 4
**If you were to recommend 3 beers to your friends based on this data, which ones would you recommend?**

In [34]:
# copy df
q4 = df.copy()

# get total reviews for each beer
reviews_df = q4.groupby('beer_name')['review_time'].size().to_frame('n_reviews').reset_index()
reviews_df['n_reviews_rank'] = reviews_df['n_reviews'].rank(ascending=False, method='dense')
q4 = pd.merge(q4, reviews_df, how='inner', on='beer_name')

# filter top 100 most reviewed beers, get top 3 based on taste
q4.loc[
    (q4['n_reviews_rank'] < 100)
].groupby('beer_name')['review_taste'].mean().nlargest(3).round(4).reset_index()

,beer_name,review_taste
0,Founders KBS (Kentucky Breakfast Stout),4.4798
1,Trappistes Rochefort 10,4.4309
2,Founders Breakfast Stout,4.3879


# Question 5
**Which beer style seems to be the favourite based on the reviews written by users? How does written reviews compare to overall review score for the beer style?**

In [35]:
# import libraries
from nltk.sentiment.vader import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()
# copy df
q5 = df.copy()
# drop rows with missing review_text
q5.dropna(subset=['review_text'], inplace=True)
q5.head(5)

,beer_ABV,beer_beerId,beer_brewerId,beer_name,beer_style,review_appearance,review_palette,review_overall,review_taste,review_profileName,review_aroma,review_text,review_time
0,5.0,47986,10325,Sausa Weizen,Hefeweizen,2.5,2.0,1.5,1.5,stcules,1.5,A lot of foam. But a lot. In the smell some ba...,2009-02-16 20:57:03
1,6.2,48213,10325,Red Moon,English Strong Ale,3.0,2.5,3.0,3.0,stcules,3.0,"Dark red color, light beige foam, average. In ...",2009-03-01 13:44:57
2,6.5,48215,10325,Black Horse Black Beer,Foreign / Export Stout,3.0,2.5,3.0,3.0,stcules,3.0,"Almost totally black. Beige foam, quite compac...",2009-03-01 14:10:04
3,5.0,47969,10325,Sausa Pils,German Pilsener,3.5,3.0,3.0,2.5,stcules,3.0,"Golden yellow color. White, compact foam, quit...",2009-02-15 19:12:25
4,7.7,64883,1075,Cauldron DIPA,American Double / Imperial IPA,4.0,4.5,4.0,4.0,johnmichaelsen,4.5,"According to the website, the style for the Ca...",2010-12-30 18:53:26


In [36]:
# get polarity scores
q5['polarity_compound'] = [sia.polarity_scores(text)['compound'] for text in q5['review_text']]

In [37]:
# top 10 favorites by written reviews based on polarity scores
written_favs = q5.groupby('beer_style', as_index=False)['polarity_compound'].mean().nlargest(columns='polarity_compound', n=10).rename(columns={'polarity_compound':'mean polarity'})    
written_favs

,beer_style,mean polarity
86,Quadrupel (Quad),0.857887
38,Dortmunder / Export Lager,0.852428
32,Braggot,0.850789
58,Flanders Red Ale,0.849444
11,American Double / Imperial Stout,0.847797
101,Wheatwine,0.842578
88,Roggenbier,0.842273
41,Eisbock,0.837225
25,Belgian Strong Dark Ale,0.832886
84,Old Ale,0.830954


In [38]:
# top 10 favorite by overall review score
review_score_favs = q5.groupby('beer_style', as_index=False)['review_overall'].mean().nlargest(columns='review_overall', n=10).rename(columns={'review_overall':'mean score'})
review_score_favs

,beer_style,mean score
63,Gueuze,4.140952
27,Berliner Weissbier,4.133976
11,American Double / Imperial Stout,4.100441
83,Oatmeal Stout,4.080804
41,Eisbock,4.079487
90,Rye Beer,4.069994
38,Dortmunder / Export Lager,4.051962
86,Quadrupel (Quad),4.049159
88,Roggenbier,4.032680
74,Lambic - Fruit,4.027070


In [39]:
# Beer Styles found in both
pd.DataFrame(list(set(written_favs['beer_style']).intersection(set(review_score_favs['beer_style']))), columns=['Beer Style'])

,Beer Style
0,American Double / Imperial Stout
1,Eisbock
2,Quadrupel (Quad)
3,Dortmunder / Export Lager
4,Roggenbier


In [40]:
import pingouin as pg
pg.corr(q5['polarity_compound'], q5['review_overall'])

,n,r,CI95%,p-val,BF10,power
pearson,528751,0.379744,"[0.38, 0.38]",0.0,inf,1.0
